<a href="https://colab.research.google.com/github/Naman1232/ML-PROJECTS/blob/main/Anomaly_Detection_using_AutoEncoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df=pd.read_csv('anomaly.csv')
print(df.head())

        Date  Power  Detector Quality
0  01-Jan-16     96         8    Good
1  02-Jan-16     96        10    Good
2  03-Jan-16     91         8    Good
3  04-Jan-16     97         9    Good
4  05-Jan-16     91        11    Good


In [2]:
print(df.groupby('Quality')['Quality'].count())

Quality
Bad      407
Good    1054
Name: Quality, dtype: int64


In [3]:
df.drop(['Date'],axis=1,inplace=True)

In [4]:
df.dropna(inplace=True,axis=1)

In [5]:
print(df.head())

   Power  Detector Quality
0     96         8    Good
1     96        10    Good
2     91         8    Good
3     97         9    Good
4     91        11    Good


In [8]:
df.loc[df.Quality == 'Good', 'Quality'] = 1
df.loc[df.Quality == 'Bad', 'Quality'] = 2


In [9]:
df.head()

,Power,Detector,Quality
0,96,8,1
1,96,10,1
2,91,8,1
3,97,9,1
4,91,11,1


In [10]:
good_mask=df['Quality']==1
bad_mask=df['Quality']==2

In [18]:
df_good=df[good_mask]
df_bad=df[bad_mask]
print(df_bad.head())

    Power  Detector
44     94         6
45     93         7
46     95         6
47     94         7
48     97         5


In [20]:
print(f"Good Count:{len(df_good)}")
print(f"Bad Count:{len(df_bad)}")

Good Count:1054
Bad Count:407


In [21]:
x_good=df_good.values
x_bad=df_bad.values

.values converts the pandas df into numpy array as neural networks work better with numpy array


In [22]:
from sklearn.model_selection import train_test_split
x_good_train,x_good_test=train_test_split(x_good,test_size=0.25,random_state=42)

In [23]:
print(f"Good train count:{len(x_good_train)}")
print(f"Good test count:{len(x_good_test)}")

Good train count:790
Good test count:264


In [25]:
from sklearn import metrics
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [28]:
model = Sequential()
model.add(Dense(10, input_dim=x_good.shape[1], activation='relu'))
model.add(Dense(3, activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(x_good.shape[1]))
model.compile(loss='mean_squared_error', optimizer='adam')
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 10)                  │              30 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │              33 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 10)                  │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 2)                   │              22 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 125 (500.00 B)

 Trainable params: 125 (500.00 B)

 Non-trainable params: 0 (0.00 B)

*   Sequential() creates a stacked neural network where layers are added one after another.
*  ✅ Takes input data with x_good.shape[1] features.
✅ Uses 10 neurons to learn initial patterns.
✅ ReLU activation helps in learning complex relationships


*   ✅ Reduces dimensionality (only 3 neurons).
✅ Learns the most important features of the data
*   Expands data back to 10 neurons, reconstructing the original input

*   ✅ Outputs a reconstruction of the original input data.
✅ No activation function (default is linear) → directly outputs values
*  x_good.shape[1] gives the number of features (columns)










In [26]:
model.fit(x_good_train,x_good_train,verbose=1,epochs=100)

pred = model.predict(x_good_test)
score1 = np.sqrt(metrics.mean_squared_error(pred,x_good_test))

pred = model.predict(x_good)
score2 = np.sqrt(metrics.mean_squared_error(pred,x_good))

pred = model.predict(x_bad)
score3 = np.sqrt(metrics.mean_squared_error(pred,x_bad))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 10)                  │              30 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │              33 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 10)                  │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 2)                   │              22 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 125 (500.00 B)

 Trainable params: 125 (500.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 5466.8101
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4935.3359
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4436.2358
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3930.4199
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3439.5735
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2876.3320
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2165.5039
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1393.8081
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 675.9223
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 215.6102
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 36.1602 
Epoch 12/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.2601 
Epoch 13/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3824 
Epoch 14/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3750 
Epoch 15/100
25/25 ━━━━━━━━━━━━━━━━━━━

In [27]:
print(f"Insample Good Score (RMSE): {score1}".format(score1))
print(f"Out of Sample Good Score (RMSE): {score2}")
print(f"Bad sample Score (RMSE): {score3}")

Insample Good Score (RMSE): 1.1581429116870643
Out of Sample Good Score (RMSE): 1.1560826567102018
Bad sample Score (RMSE): 2.9093603082566584




*   Small reconstruction error → Data is normal, no anomaly detected.
*   If Score3 > Score1 & Score2, then the model successfully detects anomalies.
The higher reconstruction error means the model struggled with "Bad" data, proving it is different from normal.

